In [1]:
from azureml.core import Workspace, Dataset, Datastore

# اتصال به Workspace
subscription_id = 'f8c5aac3-29fc-4387-858a-1f61722fb57a'
resource_group = 'forskerpl-n0ybkr-rg'
workspace_name = 'forskerpl-n0ybkr-mlw'

ws = Workspace(subscription_id=subscription_id,
               resource_group=resource_group,
               workspace_name=workspace_name)

# گرفتن datastore
datastore = Datastore.get(ws, "researcher_data")

# خواندن همه فایل‌های parquet در مسیر مشخص
dataset = Dataset.Tabular.from_parquet_files(
    path=[(datastore, 'Zahra/012026/Data/MEDS_MDPS/data/train/*.parquet')]    #     Zahra/Data-07-2025/MDP/MEDS_811/data/train
)

# تبدیل به pandas DataFrame
df = dataset.to_pandas_dataframe()
df.head(15)


/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (3.1.1) and mlflow-skinny (2.22.1) are different. This may lead to unexpected behavior. Please install the same version of both packages.
  mlflow.mismatch._check_version_mismatch()


Resolving access token for scope "https://storage.azure.com/.default" using identity of type "MANAGED".
Getting data access token with Assigned Identity (client_id=clientid) and endpoint type based on configuration
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}


,subject_id,time,code,numeric_value
0,51,NaT,GENDER//Kvinde,NaN
1,51,2018-05-29 00:00:00,D/DJ039,NaN
2,51,2020-02-19 10:57:00,P/UXUD10,NaN
3,51,2020-02-20 00:00:00,D/DK802,NaN
4,51,2020-02-24 08:25:00,P/AAF20,NaN
5,51,2020-02-24 08:25:00,P/ZZ0150,NaN
6,110,NaT,GENDER//Mand,NaN
7,110,2016-05-31 08:08:00,P/UXCD60,NaN
8,110,2016-06-06 00:00:00,D/DN189,NaN
9,110,2016-06-29 00:00:00,D/DM171B,NaN


In [2]:
len(df)

446586556

In [ ]:
import pandas as pd

# فرض بر این که فایل CSV رو داری
# df = pd.read_csv("your_file.csv")  # یا مستقیم اگر DataFrame آماده‌ست، نیازی نیست

# دسته‌بندی هر subject_id بر اساس وجود کدهای مختلف
m_patients = df[df['code'].str.startswith('M/', na=False)]['subject_id'].unique()
p_patients = df[df['code'].str.startswith('P/', na=False)]['subject_id'].unique()
d_patients = df[df['code'].str.startswith('D/', na=False)]['subject_id'].unique()
d_patients = df[df['code'].str.startswith('S/', na=False)]['subject_id'].unique()

# کل بیماران منحصربه‌فرد
all_patients = df['subject_id'].unique()

# نمایش آمار
print(f"تعداد کل بیماران: {len(all_patients)}")
print(f"تعداد بیمارانی که کد M دارند: {len(m_patients)}")
print(f"تعداد بیمارانی که کد P دارند: {len(p_patients)}")
print(f"تعداد بیمارانی که کد D دارند: {len(d_patients)}")
print(f"تعداد بیمارانی که کد S دارند: {len(d_patients)}")


In [9]:
subject_counts = df['subject_id'].value_counts()


In [10]:
subject_counts

921542     82103
106        63789
698589     63432
2016077    60873
954669     58832
           ...  
573238         2
1831164        2
131767         2
1506679        2
471874         2
Name: subject_id, Length: 2218028, dtype: int64

In [7]:
p_Num = df[df['code'].str.startswith('P/', na=False)]

In [8]:
p_Num

,subject_id,time,code,numeric_value
3,12,2017-03-16 14:00:00,P/ZZ0150,NaN
4,12,2017-03-16 14:46:00,P/ZZ3925,NaN
8,12,2017-03-22 09:01:00,P/ZZ3925,NaN
14,12,2017-03-22 11:00:00,P/BFCA21,NaN
22,12,2017-03-22 15:40:00,P/BWTT8,NaN
...,...,...,...,...
560086718,2217990,2023-11-07 17:41:00,P/BGAZ0,NaN
560086728,2217990,2023-11-07 18:10:00,P/KJEA01,NaN
560086780,2218021,2022-03-05 12:09:00,P/ZZ4300,NaN
560086781,2218021,2022-03-05 12:09:00,P/ZZ7306A,NaN


In [4]:
import pandas as pd

# پیدا کردن سطرهایی که فقط codeهای نوع /P دارن
only_p = df[df['code'].str.startswith('P/', na=False)]

# بیماران با فقط /P کد
subject_ids_only_p = only_p['subject_id'].unique()

# حالا بیماران با codeهای غیر از /P
not_p = df[~df['code'].str.startswith('P/', na=False)]
subject_ids_with_non_p = set(not_p['subject_id'].unique())

# حذف بیمارانی که فقط /P دارن
only_p_ids_to_exclude = [sid for sid in subject_ids_only_p if sid not in subject_ids_with_non_p]

print("Number of patients with only porcedure code: ", only_p_ids_to_exclude)


Number of patients with only porcedure code:  []


In [2]:
df_filtered = df[~df['code'].str.startswith('P/', na=False)]

In [11]:
subject_counts_MD = df_filtered['subject_id'].value_counts()

In [12]:
subject_counts_MD

921542     77721
698589     61351
106        60295
2016077    58264
954669     55553
           ...  
274030         2
1414395        2
1154835        2
730877         2
2049904        2
Name: subject_id, Length: 2218028, dtype: int64

In [13]:
subject_counts_df = subject_counts.reset_index()
subject_counts_df.columns = ['subject_id', 'original_count']

subject_counts_MD_df = subject_counts_MD.reset_index()
subject_counts_MD_df.columns = ['subject_id', 'new_count']


In [15]:
import pandas as pd
comparison_df = pd.merge(subject_counts_df, subject_counts_MD_df, on='subject_id', how='outer')


In [19]:
comparison_df['difference'] =  comparison_df['original_count'] - comparison_df['new_count']


In [17]:
comparison_df = comparison_df.sort_values(by='difference', ascending=False)


In [20]:
comparison_df

,subject_id,original_count,new_count,difference
2218027,471874,2,2,0
2032815,1516003,13,13,0
2032726,1048227,13,13,0
2133552,588531,7,7,0
2032722,1108999,13,13,0
...,...,...,...,...
0,921542,82103,77721,4382
8577,1887674,4651,65,4586
5,1964088,57148,52243,4905
30,714050,42992,37665,5327


In [21]:
unchanged_count = (comparison_df['difference'] == 0).sum()
print("unchanged_count", unchanged_count)


unchanged_count 91175


In [22]:
comparison_df['abs_diff'] = comparison_df['difference'].abs()
most_changed = comparison_df.sort_values(by='abs_diff', ascending=False)


In [23]:
print(most_changed.head(10))


      subject_id  original_count  new_count  difference  abs_diff
7        1291159           56179      49845        6334      6334
30        714050           42992      37665        5327      5327
5        1964088           57148      52243        4905      4905
8577     1887674            4651         65        4586      4586
0         921542           82103      77721        4382      4382
16       1169555           49087      45012        4075      4075
49          9512           36660      33002        3658      3658
218      2109519           20019      16394        3625      3625
1            106           63789      60295        3494      3494
4         954669           58832      55553        3279      3279


In [24]:
changed_df = comparison_df[comparison_df['difference'] != 0]
min_new_count = changed_df['new_count'].min()
lowest_new_count_patients = changed_df[changed_df['new_count'] == min_new_count]


In [26]:
lowest_new_count_patients = lowest_new_count_patients.rename(
    columns={
        'original_count': 'MDP codes',
        'new_count': 'MD codes'
    }
)


In [27]:
lowest_new_count_patients

,subject_id,MDP codes,MD codes,difference,abs_diff
2194567,2087106,3,2,1,1
2191218,1328943,3,2,1,1
2208457,1144261,3,2,1,1
2203878,426371,3,2,1,1
2203490,715600,3,2,1,1
...,...,...,...,...,...
1818262,816167,21,2,19,19
1815771,2076088,21,2,19,19
1778495,1896771,22,2,20,20
1738622,1181764,24,2,22,22


.str.upper() شرط را case-insensitive می‌کند

بل از فیلتر، ستون را به pd.StringDtype() تبدیل می‌کند؛ این کار رفتار .str را پایدار و قابل پیش‌بینی می‌کند (<NA> به‌جای NaN)



In [3]:
# کل بیماران منحصربه‌فرد
all_patients = df_filtered['subject_id'].unique()
all_patients
# نمایش آمار

array([     12,      37,     124, ..., 2217959, 2217990, 2218021])

In [4]:
len(all_patients)

2218028

In [5]:
len(df_filtered)

454591284

In [6]:
import pandas as pd
import numpy as np

# امن‌تر: اگر code نال یا غیررشته‌ای بود اذیت نکنه
df['code'] = df['code'].astype('string')
df_filtered = df[~df['code'].str.upper().str.startswith('P/', na=False)].copy()
print("kept rows MD:", len(df_filtered), " / total:", len(df))


kept rows MD: 454591284  / total: 560086801


In [4]:
import pandas as pd
import numpy as np

# اطمینان از نوع‌ها (برای خروجی تمیز و بدون خطا)
df_filtered = df_filtered.copy()
df_filtered['subject_id']    = pd.to_numeric(df_filtered['subject_id'], errors='coerce').astype('Int64')
df_filtered['numeric_value'] = pd.to_numeric(df_filtered['numeric_value'], errors='coerce').astype('float32')
df_filtered['time']          = pd.to_datetime(df_filtered['time'], errors='coerce', utc=False)

# ردیف‌های بدون subject_id را حذف کنیم (نمی‌توان شارد کرد)
df_filtered = df_filtered.dropna(subset=['subject_id']).copy()
df_filtered['subject_id'] = df_filtered['subject_id'].astype('int64')

# ۳۶ شارد: هر بیمار فقط در یک فایل (mod 36)
N_SHARDS = 45
df_filtered['__shard__'] = (df_filtered['subject_id'] % N_SHARDS).astype('int16')

print("rows to write:", len(df_filtered))


rows to write: 454591284


In [7]:
import numpy as np
import os

N_SHARDS = 45    # 36 when we have split
OUT_DIR = "./_Wholedata_withoutP_sharded"
os.makedirs(OUT_DIR, exist_ok=True)

cols_out = ['subject_id', 'time', 'code', 'numeric_value']

# فرض: subject_id قبلاً int64 شده و NaNها حذف شده‌اند (طبق سلول قبلی‌ات)
sid_mod = (df_filtered['subject_id'].to_numpy(dtype=np.int64, copy=False) % N_SHARDS)

written = 0
for k in range(N_SHARDS):
    mask = (sid_mod == k)                 # بدون ستون اضافی، فقط یک آرایهٔ NumPy
    part = df_filtered.loc[mask, cols_out].sort_values(['subject_id','time'])
    # اگر می‌خوای حتماً ۳۶ فایل 0..35 داشته باشی حتی اگه خالی باشن:
    # if part.empty:
    #     part = part.iloc[0:0]  # فایل صفر-سطر با همان ستون‌ها
    part.to_parquet(os.path.join(OUT_DIR, f"{k}.parquet"),
                    engine="pyarrow", compression="snappy", index=False)
    written += 1

print(f"Done. wrote {written} parquet files into {OUT_DIR}")


Done. wrote 45 parquet files into ./_Wholedata_withoutP_sharded


In [8]:
import pyarrow.parquet as pq

def parquet_num_rows(path):
    pf = pq.ParquetFile(path)
    md = pf.metadata
    return sum(md.row_group(i).num_rows for i in range(md.num_row_groups))

total = 0
for f in sorted(p for p in os.listdir(OUT_DIR) if p.endswith('.parquet')):
    n = parquet_num_rows(os.path.join(OUT_DIR, f))
    total += n
    print(f, "rows:", n)
print("TOTAL rows:", total)


0.parquet rows: 10027797
1.parquet rows: 10068068
10.parquet rows: 10163953
11.parquet rows: 10195177
12.parquet rows: 9967646
13.parquet rows: 10251231
14.parquet rows: 10448621
15.parquet rows: 9943901
16.parquet rows: 10262618
17.parquet rows: 10180367
18.parquet rows: 10123911
19.parquet rows: 10392829
2.parquet rows: 10038096
20.parquet rows: 9848138
21.parquet rows: 9963469
22.parquet rows: 10193682
23.parquet rows: 10286489
24.parquet rows: 10058057
25.parquet rows: 10214272
26.parquet rows: 9974841
27.parquet rows: 10231763
28.parquet rows: 10009157
29.parquet rows: 10077745
3.parquet rows: 9971065
30.parquet rows: 10089686
31.parquet rows: 10140589
32.parquet rows: 10340793
33.parquet rows: 10065581
34.parquet rows: 9713042
35.parquet rows: 10333562
36.parquet rows: 9990844
37.parquet rows: 10257693
38.parquet rows: 9939632
39.parquet rows: 10038701
4.parquet rows: 9880914
40.parquet rows: 10136903
41.parquet rows: 10079823
42.parquet rows: 10338758
43.parquet rows: 9879109
44

In [10]:
paths = Dataset.File.from_files(path=[(datastore, f"{DST_PREFIX}/*.parquet")]).to_path()
print("Found in datastore:", len(paths))
print(paths[:5])


{'infer_column_types': 'False', 'activity': 'to_path'}
{'infer_column_types': 'False', 'activity': 'to_path', 'activityApp': 'FileDataset'}
Found in datastore: 36
['/0.parquet', '/1.parquet', '/10.parquet', '/11.parquet', '/12.parquet']
